# GeoMine-XNet: real-data training on free Colab

This notebook trains the paper-informed SAR/optical model from **your actual reviewed 10m dataset**. No sample scenes, labels, weights, metrics, or paid compute are supplied. Colab GPU availability/runtime duration varies; choose **Runtime → Change runtime type → GPU**. This does not deploy a trained detector automatically.

The data gate checks provenance, label review, geographic separation, duplicates, shapes and valid coverage. It cannot certify label correctness. Do not substitute static mine extents for temporal change labels or upscale public 30m radar and call it 10m ground truth.


In [ ]:
import sys, subprocess
import torch
assert torch.cuda.is_available(), "Select a GPU runtime. No paid runtime is required by this notebook."
print({"gpu": torch.cuda.get_device_name(0), "pytorch": torch.__version__})


## 1. Load the actual source and your data

Upload the secret-free `geomine-xnet-source.zip` supplied with the project. Dataset archives should be prepared using the documented operator workflow, with `manifest.json` at the ZIP root. Do not upload production `.env` files, service-account keys, or a database backup.

Check that uploaded code is your reviewed project source before running it.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload only the reviewed source ZIP in this cell."
source_root = Path("/content/geomine-source")
source_root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.infolist():
        target = (source_root / item.filename).resolve()
        assert target.is_relative_to(source_root.resolve()), "Unsafe source archive path"
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Symlink not allowed"
    archive.extractall(source_root)
backend = source_root / "geomine-xnet" / "backend"
assert (backend / "app/ml/train.py").is_file()
os.chdir(backend)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-ml.txt"], check=True)


## 2. Persist genuine artifacts in your Drive

Mount your own Drive and set `DATASET_ZIP` to the real reviewed dataset ZIP. Keep the input archive and training outputs in your private Drive. A fresh immutable run folder is created each time.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount("/content/drive")
DATASET_ZIP = Path("/content/drive/MyDrive/GeoMine/real-reviewed-dataset.zip")  # set your actual path
assert DATASET_ZIP.is_file(), "Supply a real reviewed dataset archive; no fallback data is created."
run_name = datetime.now(timezone.utc).strftime("run-%Y%m%d-%H%M%S")
dataset = Path("/content") / (run_name + "-dataset")
output = Path("/content/drive/MyDrive/GeoMine/checkpoints") / run_name
subprocess.run([sys.executable, "-m", "app.ml.artifacts", "import-dataset", "--archive", str(DATASET_ZIP), "--destination", str(dataset)], check=True)
subprocess.run([sys.executable, "-m", "app.ml.audit", "--dataset", str(dataset), "--output", str(dataset / "audit.json")], check=True)


## 3. Train and select using validation only

Batch size 1 with accumulation of 4 limits GPU memory while retaining an effective batch of 4. Mixed precision is enabled. Set the epoch count for your reviewed experiment; 50 matches the paper's stated epoch count but is not a guarantee a free session can finish.

The best validation checkpoint and history are written to Drive after each epoch. If a session ends, `WARM_START` may point to that previous `model.safetensors`. A warm start uses a **new optimizer and schedule**; it is not an exact resume. Use a new run folder and record the previous artifact. Training does not evaluate the test split unless explicitly requested in a frozen final experiment.


In [ ]:
EPOCHS = 50
WARM_START = None  # optional Path to your own previous genuine model.safetensors
command = [sys.executable, "-m", "app.ml.train", "--dataset", str(dataset), "--output", str(output), "--epochs", str(EPOCHS), "--batch-size", "1", "--accumulate", "4", "--amp", "--device", "cuda"]
FINAL_FROZEN_EXPERIMENT = False  # set True only after freezing choices using validation
if FINAL_FROZEN_EXPERIMENT:
    command += ["--evaluate-test"]
if WARM_START:
    command += ["--warm-start", str(WARM_START)]
subprocess.run(command, check=True)


## 4. Inspect measured results

These plots read the actual saved training history. They do not create substitute values if training has not run. A passed software/data audit and good validation metrics do not establish performance on Ghana/Amazon field conditions.


In [ ]:
import json
import matplotlib.pyplot as plt
history = json.loads((output / "history.json").read_text())
assert history, "No recorded training history."
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([row["epoch"] for row in history], [row["train_loss"] for row in history])
axes[0].set(xlabel="Completed epoch", ylabel="Measured training loss")
for metric in ("precision", "recall", "f1", "foreground_iou"):
    axes[1].plot([row["epoch"] for row in history], [row["validation"][metric] for row in history], label=metric)
axes[1].set(xlabel="Completed epoch", ylabel="Validation score", ylim=(0, 1))
axes[1].legend()
plt.show()
manifest = json.loads((output / "manifest.json").read_text())
print({key: manifest.get(key) for key in ["validation_metrics", "test_metrics", "release_approved", "epoch", "effective_batch_size"]})


## 5. Review before installation

Declare release criteria before final test evaluation. Check field relevance, per-site mistakes, cloud/season cases, false positives, and labels with a human reviewer. The operator's `approve-model` command requires explicit criteria and a written review. Only then use `install-model` into a new immutable version directory.

The deployed public radar source currently runs at 30m; this 10m model stays disabled until compatible data access is configured. Never install random weights or report this notebook as trained until a real run and held-out report exist.

References: [Colab availability/limits](https://research.google.com/colaboratory/faq.html), [EuroMineNet optical benchmark](https://rodare.hzdr.de/record/4656), [global static mining-site labels](https://zenodo.org/records/14195737). Those public datasets have different modalities/label semantics and are not drop-in GeoMine-XNet temporal fusion ground truth.
